# LeetCode 1817 — Finding the Users Active Minutes bằng Python

Link đề: https://leetcode.com/problems/finding-the-users-active-minutes/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def findingUsersActiveMinutes(self, logs: List[List[int]], k: int) -> List[int]:
        ...
```

Lộ trình: **quét lại toàn bộ log cho từng người dùng** `O(U · L)` → **sắp xếp log rồi đếm theo nhóm** `O(L log L)` → **`dict` ánh xạ người dùng → `set` phút** `O(L)` (lời giải nộp) → **`set` các cặp `(id, phút)` + `Counter`**.

## 1. Mục tiêu học

1. Gom dữ liệu theo khóa bằng **`defaultdict(set)`** — vừa nhóm theo người dùng, vừa **khử trùng lặp** phút.
2. Phân biệt "số **hành động**" với "số **phút khác nhau**" — đúng định nghĩa UAM.
3. Dựng mảng kết quả **đánh số từ 1** trong list Python đánh số từ 0 (`ans[uam - 1]`), và độ dài đúng `k`.
4. So sánh các cách nhóm: quét lại, sắp xếp + gom khúc, băm.
5. Kiểm chứng bằng vét cạn ngẫu nhiên có seed, đúng ràng buộc (`k ≥ UAM lớn nhất`).

## 2. Đề bài & ví dụ

Cho log hoạt động của người dùng trên LeetCode và số nguyên `k`. Log là mảng 2 chiều `logs`, mỗi `logs[i] = [ID_i, time_i]` nghĩa là người dùng `ID_i` có một hành động tại phút `time_i`.

Nhiều người có thể hoạt động cùng lúc; một người có thể có nhiều hành động trong cùng một phút.

**UAM** (user active minutes) của một người = số **phút khác nhau** mà người đó có hành động (một phút chỉ tính một lần dù có nhiều hành động).

Tính mảng `answer` **đánh số từ 1**, độ dài `k`: với mỗi `1 ≤ j ≤ k`, `answer[j]` = **số người dùng có UAM bằng `j`**. (Trong code trả về list 0-indexed: phần tử thứ `j − 1` ứng với `answer[j]`.)

**Ví dụ 1:** `logs = [[0,5],[1,2],[0,2],[0,5],[1,3]], k = 5` → `[0,2,0,0,0]`. Người 0: phút `{5, 2}` ⇒ UAM 2; người 1: `{2, 3}` ⇒ UAM 2.

**Ví dụ 2:** `logs = [[1,1],[2,2],[2,3]], k = 4` → `[1,1,0,0]`.

**Ràng buộc:** `1 ≤ logs.length ≤ 10⁴`, `0 ≤ ID_i ≤ 10⁹`, `1 ≤ time_i ≤ 10⁵`, `k` nằm trong `[UAM lớn nhất, 10⁵]`.

In [1]:
import random
import time
from collections import Counter, defaultdict
from itertools import groupby
from typing import List

examples = [
    ([[0, 5], [1, 2], [0, 2], [0, 5], [1, 3]], 5, [0, 2, 0, 0, 0]),
    ([[1, 1], [2, 2], [2, 3]], 4, [1, 1, 0, 0]),
]
for logs, k, want in examples:
    print(logs, "k =", k, "->", want)

[[0, 5], [1, 2], [0, 2], [0, 5], [1, 3]] k = 5 -> [0, 2, 0, 0, 0]
[[1, 1], [2, 2], [2, 3]] k = 4 -> [1, 1, 0, 0]


## 3. Kiến thức nền

### 3.1 `set` — tập hợp không trùng lặp

`add` một phần tử đã có thì không thay đổi gì; `len(s)` là số phần tử **khác nhau**.

In [2]:
s = set()
for t in [5, 2, 5]:
    s.add(t)
print(s, len(s))

{2, 5} 2


### 3.2 `defaultdict(set)` — gom nhóm

Truy cập khóa chưa có sẽ tự tạo `set()` rỗng. Một dòng `d[user].add(minute)` vừa nhóm vừa khử trùng.

In [3]:
d = defaultdict(set)
for uid, t in [[0, 5], [1, 2], [0, 2], [0, 5], [1, 3]]:
    d[uid].add(t)
print(dict(d))

{0: {2, 5}, 1: {2, 3}}


### 3.3 List đánh số từ 0 cho mảng "1-indexed"

`answer[j]` (1-indexed) nằm ở `ans[j − 1]`. Tạo `ans = [0] * k` để có đủ `k` ô, kể cả các ô bằng 0.

In [4]:
k = 5
ans = [0] * k
uam = 2
ans[uam - 1] += 1
print(ans)

[0, 1, 0, 0, 0]


### 3.4 `Counter` để đếm tần suất của tần suất

`Counter(iterable)` đếm số lần xuất hiện. Ở đây ta "đếm hai tầng": tầng 1 đếm số phút mỗi người, tầng 2 đếm số người theo UAM.

In [5]:
uam_of_user = {0: 2, 1: 2, 7: 1}
print(Counter(uam_of_user.values()))

Counter({2: 2, 1: 1})


### 3.5 Sắp xếp list cặp và `groupby`

`sorted(logs)` sắp theo `ID` rồi theo `time`. `groupby(..., key=lambda p: p[0])` gom các log liên tiếp cùng `ID`.

In [6]:
logs = [[0, 5], [1, 2], [0, 2], [0, 5], [1, 3]]
for uid, grp in groupby(sorted(logs), key=lambda p: p[0]):
    print(uid, [t for _, t in grp])

0 [2, 5, 5]
1 [2, 3]


## 4. Ý tưởng nền tảng / trực giác

Tính tay Ví dụ 1:

| log | người 0 (phút) | người 1 (phút) |
|---|---|---|
| `[0,5]` | {5} | |
| `[1,2]` | {5} | {2} |
| `[0,2]` | {5, 2} | {2} |
| `[0,5]` | {5, 2} (5 đã có) | {2} |
| `[1,3]` | {5, 2} | {2, 3} |

UAM: người 0 = 2, người 1 = 2 ⇒ có 2 người UAM = 2 ⇒ `answer[2] = 2` ⇒ list `[0, 2, 0, 0, 0]`.

Hai bước tách bạch: (1) **mỗi người có bao nhiêu phút khác nhau**; (2) **đếm số người theo con số đó**.

## 5. Cách 1 — Brute Force / baseline: quét lại log cho từng người

**Ý tưởng:** lấy danh sách người dùng khác nhau; với mỗi người, quét **toàn bộ** log để gom phút của họ vào một list, khử trùng bằng cách kiểm "đã có trong list chưa" (không dùng `set`).

In [7]:
def uam_brute(logs, k):
    users = []
    for uid, _ in logs:
        if uid not in users:                 # O(U) mỗi lần
            users.append(uid)
    ans = [0] * k
    for u in users:
        minutes = []
        for uid, t in logs:                  # quét lại toàn bộ log: O(L)
            if uid == u and t not in minutes:
                minutes.append(t)
        ans[len(minutes) - 1] += 1
    return ans

for logs, k, want in examples:
    print(uam_brute(logs, k), "| kỳ vọng", want)

[0, 2, 0, 0, 0] | kỳ vọng [0, 2, 0, 0, 0]
[1, 1, 0, 0] | kỳ vọng [1, 1, 0, 0]


**Độ phức tạp:** gọi `L = len(logs)`, `U` = số người. Tìm người dùng `O(L·U)`; mỗi người quét `L` log, mỗi log kiểm `t not in minutes` `O(UAM)` ⇒ worst `O(U·L + L²)`. Với `L = 10⁴` vẫn chạy được (cỡ `10⁸` trong trường hợp xấu), nhưng lãng phí: mỗi log bị đọc lại `U` lần.

**Nút thắt:** không có cấu trúc "nhóm theo người" và "tập phút" — thêm hai bảng băm là xong.

In [8]:
rng = random.Random(0)
logs_bad = [[i, 1] for i in range(4000)]            # 4000 người, mỗi người 1 log -> quét lại 4000 lần
t = time.perf_counter(); uam_brute(logs_bad, 1); dt = time.perf_counter() - t
print(f"L=U=4000: {dt:.2f}s")

L=U=4000: 0.28s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** nhóm log theo `ID` **một lần** bằng bảng băm `ID → set(phút)` — `set` tự khử trùng phút.

**Insight 2:** UAM của người `u` là `len(d[u])`. Sau đó chỉ cần `ans[len(d[u]) − 1] += 1`.

**Quy tắc:**
- *Nếu* gặp log `[u, t]` *thì* `d[u].add(t)`.
- *Sau khi* đọc hết log, *với mỗi* `u`: `ans[len(d[u]) − 1] += 1`.

Ràng buộc bảo đảm `k ≥ UAM lớn nhất` nên chỉ số `len − 1 ≤ k − 1` luôn hợp lệ, và UAM ≥ 1 (người xuất hiện trong log thì có ít nhất một phút) nên chỉ số ≥ 0.

**Invariant:** sau khi đọc `t` log đầu, `d[u]` = tập phút khác nhau của `u` trong `t` log đó.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp log rồi đếm theo khúc (`O(L log L)`)

Sau `sorted(logs)`, log của cùng một người nằm liền nhau và phút đã tăng dần ⇒ phút trùng nằm cạnh nhau. Đếm số phút khác nhau = số vị trí mà cặp `(id, time)` khác cặp trước.

In [9]:
def uam_sort(logs, k):
    ans = [0] * k
    srt = sorted(logs)
    cur_user, cur_uam, prev = None, 0, None
    for uid, t in srt:
        if uid != cur_user:
            if cur_user is not None:
                ans[cur_uam - 1] += 1
            cur_user, cur_uam, prev = uid, 0, None
        if t != prev:
            cur_uam += 1
            prev = t
    ans[cur_uam - 1] += 1                     # chốt người cuối
    return ans

for logs, k, want in examples:
    print(uam_sort(logs, k), want)

[0, 2, 0, 0, 0] [0, 2, 0, 0, 0]
[1, 1, 0, 0] [1, 1, 0, 0]


**Độ phức tạp:** `O(L log L)` sắp xếp + `O(L)` đếm; bộ nhớ `O(L)` cho bản sắp xếp. Ưu điểm: không cần băm, kết quả trung gian có thứ tự (hữu ích nếu cần in theo người dùng).

### 7.2 Cách 3 — `dict` người dùng → `set` phút (`O(L)`) — lời giải chuẩn

In [10]:
def uam_dict_demo(logs, k):
    d = defaultdict(set)
    for uid, t in logs:
        before = len(d[uid])
        d[uid].add(t)
        print(f"log [{uid}, {t}]: phút của {uid} = {sorted(d[uid])}{'  (trùng, bỏ qua)' if len(d[uid]) == before else ''}")
    ans = [0] * k
    for uid, ts in d.items():
        ans[len(ts) - 1] += 1
        print(f"người {uid}: UAM = {len(ts)} -> ans = {ans}")
    return ans

uam_dict_demo(*examples[0][:2])

log [0, 5]: phút của 0 = [5]
log [1, 2]: phút của 1 = [2]
log [0, 2]: phút của 0 = [2, 5]
log [0, 5]: phút của 0 = [2, 5]  (trùng, bỏ qua)
log [1, 3]: phút của 1 = [2, 3]
người 0: UAM = 2 -> ans = [0, 1, 0, 0, 0]
người 1: UAM = 2 -> ans = [0, 2, 0, 0, 0]


[0, 2, 0, 0, 0]

In [11]:
class SolutionDict:
    def findingUsersActiveMinutes(self, logs: List[List[int]], k: int) -> List[int]:
        d = defaultdict(set)
        for uid, t in logs:
            d[uid].add(t)
        ans = [0] * k
        for ts in d.values():
            ans[len(ts) - 1] += 1
        return ans

for logs, k, want in examples:
    print(SolutionDict().findingUsersActiveMinutes(logs, k) == want)

True
True


**Độ phức tạp:** `O(L + k)` thời gian (tạo `ans` dài `k`), `O(L + k)` bộ nhớ.

### 7.3 Cách 4 — `set` các cặp `(id, phút)` + hai tầng `Counter`

Khử trùng **cả cặp** trước: `pairs = {(u, t) for u, t in logs}`. Khi đó UAM của `u` = số cặp có `id = u` ⇒ `Counter(u for u, _ in pairs)`. Tầng 2: đếm số người theo UAM.

In [12]:
class SolutionPairs:
    def findingUsersActiveMinutes(self, logs: List[List[int]], k: int) -> List[int]:
        pairs = {(u, t) for u, t in logs}
        uam = Counter(u for u, _ in pairs)          # người -> UAM
        by_uam = Counter(uam.values())              # UAM -> số người
        return [by_uam[j] for j in range(1, k + 1)]

for logs, k, want in examples:
    print(SolutionPairs().findingUsersActiveMinutes(logs, k) == want)

True
True


**Độ phức tạp:** `O(L + k)`; viết gần như "khai báo" (declarative), dễ đọc khi đã quen `Counter`.

## 8. TRACE từng bước

Trace Cách 2 (sắp xếp) trên Ví dụ 1. Sau sắp xếp: `[[0,2],[0,5],[0,5],[1,2],[1,3]]`.

| log | `cur_user` | `prev` | hành động | `cur_uam` | `ans` |
|---|---|---|---|---|---|
| `[0,2]` | 0 (mới) | — | phút mới | 1 | `[0,0,0,0,0]` |
| `[0,5]` | 0 | 2 | phút mới | 2 | |
| `[0,5]` | 0 | 5 | **trùng** | 2 | |
| `[1,2]` | 1 (mới) | — | chốt người 0: `ans[1] += 1` | 1 | `[0,1,0,0,0]` |
| `[1,3]` | 1 | 2 | phút mới | 2 | |
| hết | | | chốt người 1: `ans[1] += 1` | | `[0,2,0,0,0]` |

In [13]:
def trace_sort(logs, k):
    ans = [0] * k
    srt = sorted(logs)
    print("sau sắp xếp:", srt)
    cur_user, cur_uam, prev = None, 0, None
    for uid, t in srt:
        if uid != cur_user:
            if cur_user is not None:
                ans[cur_uam - 1] += 1
                print(f"  chốt người {cur_user}: UAM={cur_uam} -> ans={ans}")
            cur_user, cur_uam, prev = uid, 0, None
        if t != prev:
            cur_uam += 1; prev = t
            print(f"  [{uid},{t}] phút mới -> UAM tạm {cur_uam}")
        else:
            print(f"  [{uid},{t}] trùng phút, bỏ qua")
    ans[cur_uam - 1] += 1
    print(f"  chốt người {cur_user}: UAM={cur_uam} -> ans={ans}")
    return ans

trace_sort(*examples[0][:2])

sau sắp xếp: [[0, 2], [0, 5], [0, 5], [1, 2], [1, 3]]
  [0,2] phút mới -> UAM tạm 1
  [0,5] phút mới -> UAM tạm 2
  [0,5] trùng phút, bỏ qua
  chốt người 0: UAM=2 -> ans=[0, 1, 0, 0, 0]
  [1,2] phút mới -> UAM tạm 1
  [1,3] phút mới -> UAM tạm 2
  chốt người 1: UAM=2 -> ans=[0, 2, 0, 0, 0]


[0, 2, 0, 0, 0]

In [14]:
trace_sort(*examples[1][:2])

sau sắp xếp: [[1, 1], [2, 2], [2, 3]]
  [1,1] phút mới -> UAM tạm 1
  chốt người 1: UAM=1 -> ans=[1, 0, 0, 0]
  [2,2] phút mới -> UAM tạm 1
  [2,3] phút mới -> UAM tạm 2
  chốt người 2: UAM=2 -> ans=[1, 1, 0, 0]


[1, 1, 0, 0]

## 9. Code nộp LeetCode

In [15]:
from collections import defaultdict
from typing import List

class Solution:
    def findingUsersActiveMinutes(self, logs: List[List[int]], k: int) -> List[int]:
        minutes = defaultdict(set)            # ID -> tập phút khác nhau
        for uid, t in logs:
            minutes[uid].add(t)
        ans = [0] * k                         # ans[j-1] = số người có UAM = j
        for ts in minutes.values():
            ans[len(ts) - 1] += 1
        return ans

for logs, k, want in examples:
    assert Solution().findingUsersActiveMinutes(logs, k) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bước nhóm:** theo invariant ở mục 6, sau khi đọc hết log, `minutes[u]` là đúng tập các phút mà người `u` có ít nhất một hành động ⇒ `len(minutes[u])` = UAM(u) theo định nghĩa (mỗi phút đếm một lần nhờ tính chất của `set`). Người không xuất hiện trong log không có khóa — đúng, vì đề chỉ đếm người có log (và UAM ≥ 1).

**Bước đếm:** mỗi người đóng góp đúng `+1` vào ô `UAM(u) − 1`, tức `answer[UAM(u)]` (1-indexed) ⇒ `answer[j]` = số người có UAM bằng `j`. Ràng buộc `1 ≤ UAM ≤ k` bảo đảm chỉ số hợp lệ. Ô không ai có UAM tương ứng giữ giá trị 0.

**Cách 2:** sau sắp xếp theo `(id, time)`, các log cùng người liền nhau và các phút bằng nhau liền nhau ⇒ số lần `t != prev` trong khúc = số phút khác nhau.

## 11. Phân tích Time / Space

Gọi `L = len(logs) ≤ 10⁴`, `U` = số người, `k ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Quét lại | `O(U·L + L·UAM)` | `O(L + k)` |
| 2. Sắp xếp | `O(L log L + k)` | `O(L + k)` |
| 3. dict → set | `O(L + k)` trung bình | `O(L + k)` |
| 4. set cặp + Counter | `O(L + k)` trung bình | `O(L + k)` |

In [16]:
rng = random.Random(1817)
logs_big = [[rng.randint(0, 2000), rng.randint(1, 10**5)] for _ in range(10**4)]
k_big = 10**5
res = []
for name, f in [("Cách 1", uam_brute), ("Cách 2", uam_sort), ("Cách 3", SolutionDict().findingUsersActiveMinutes),
                ("Cách 4", SolutionPairs().findingUsersActiveMinutes), ("Nộp", Solution().findingUsersActiveMinutes)]:
    t = time.perf_counter(); r = f(logs_big, k_big); res.append(r)
    print(f"{name}: {1000*(time.perf_counter()-t):.1f} ms | 6 ô đầu {r[:6]}")
assert all(r == res[0] for r in res)

Cách 1: 405.6 ms | 6 ô đầu [70, 167, 277, 368, 334, 307]
Cách 2: 4.0 ms | 6 ô đầu [70, 167, 277, 368, 334, 307]
Cách 3: 2.8 ms | 6 ô đầu [70, 167, 277, 368, 334, 307]
Cách 4: 12.6 ms | 6 ô đầu [70, 167, 277, 368, 334, 307]
Nộp: 1.6 ms | 6 ô đầu [70, 167, 277, 368, 334, 307]


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [17]:
sols = [Solution(), SolutionDict(), SolutionPairs()]
edge = examples + [
    ([[0, 1]], 1, [1]),                              # một log
    ([[0, 1], [0, 1], [0, 1]], 1, [1]),              # cùng phút lặp lại
    ([[5, 1], [5, 2], [5, 3]], 3, [0, 0, 1]),        # UAM == k (biên trên chỉ số)
    ([[1, 7], [2, 7], [3, 7]], 2, [3, 0]),           # nhiều người cùng phút
    ([[10**9, 1], [0, 1], [10**9, 2]], 4, [1, 1, 0, 0]),
]
for logs, k, want in edge:
    for s in sols:
        assert s.findingUsersActiveMinutes(logs, k) == want, (logs, k, s)
    assert uam_brute(logs, k) == want == uam_sort(logs, k)
    print("OK", logs, k, "->", want)

OK [[0, 5], [1, 2], [0, 2], [0, 5], [1, 3]] 5 -> [0, 2, 0, 0, 0]
OK [[1, 1], [2, 2], [2, 3]] 4 -> [1, 1, 0, 0]
OK [[0, 1]] 1 -> [1]
OK [[0, 1], [0, 1], [0, 1]] 1 -> [1]
OK [[5, 1], [5, 2], [5, 3]] 3 -> [0, 0, 1]
OK [[1, 7], [2, 7], [3, 7]] 2 -> [3, 0]
OK [[1000000000, 1], [0, 1], [1000000000, 2]] 4 -> [1, 1, 0, 0]


### 12.2 Ngẫu nhiên có seed: so với vét cạn

Sinh log với ít người, ít phút (nhiều trùng lặp); chọn `k` ngẫu nhiên trong `[UAM lớn nhất, UAM lớn nhất + 5]` để đúng ràng buộc.

In [18]:
rng = random.Random(42)
for _ in range(1500):
    L = rng.randint(1, 30)
    logs = [[rng.randint(0, 5), rng.randint(1, 8)] for _ in range(L)]
    max_uam = max(len({t for u2, t in logs if u2 == u}) for u, _ in logs)
    k = rng.randint(max_uam, max_uam + 5)
    want = uam_brute(logs, k)
    for s in sols:
        assert s.findingUsersActiveMinutes(logs, k) == want, (logs, k)
    assert uam_sort(logs, k) == want
    assert sum(want) == len({u for u, _ in logs})      # tổng = số người
print("1500 case ngẫu nhiên khớp vét cạn; tổng ans luôn = số người")

1500 case ngẫu nhiên khớp vét cạn; tổng ans luôn = số người


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm số hành động thay vì số phút khác nhau

In [19]:
def wrong_count_actions(logs, k):
    d = defaultdict(list)
    for uid, t in logs:
        d[uid].append(t)                        # SAI: list giữ cả phút trùng
    ans = [0] * k
    for ts in d.values():
        if len(ts) <= k:
            ans[len(ts) - 1] += 1
    return ans

logs, k, want = examples[0]
print("sai:", wrong_count_actions(logs, k), "| đúng:", want)

sai: [0, 1, 1, 0, 0] | đúng: [0, 2, 0, 0, 0]


**Vì sao sai:** người 0 có 3 hành động nhưng chỉ 2 phút khác nhau (phút 5 lặp). **Sửa:** dùng `set`.

### Lỗi 2 — Quên trừ 1 khi ghi vào list 0-indexed

In [20]:
def wrong_index(logs, k):
    d = defaultdict(set)
    for uid, t in logs:
        d[uid].add(t)
    ans = [0] * k
    for ts in d.values():
        ans[len(ts)] += 1                       # SAI: lệch một ô
    return ans

for logs, k, want in [examples[0], ([[5, 1], [5, 2], [5, 3]], 3, [0, 0, 1])]:
    try:
        print("sai:", wrong_index(logs, k), "| đúng:", want)
    except IndexError as e:
        print("IndexError:", e, "| đúng:", want)

sai: [0, 0, 2, 0, 0] | đúng: [0, 2, 0, 0, 0]
IndexError: list index out of range | đúng: [0, 0, 1]


**Vì sao sai:** `answer` của đề đánh số từ 1, list Python từ 0 ⇒ UAM `j` nằm ở `ans[j − 1]`. Không trừ thì kết quả lệch sang phải, và khi UAM = `k` thì vượt chỉ số.

### Lỗi 3 — Gộp phút của mọi người vào một tập chung

In [21]:
def wrong_global_set(logs, k):
    seen = set()
    cnt = defaultdict(int)
    for uid, t in logs:
        if t not in seen:                       # SAI: phút đã thuộc người khác thì bị bỏ
            seen.add(t)
            cnt[uid] += 1
    ans = [0] * k
    for v in cnt.values():
        ans[v - 1] += 1
    return ans

logs, k, want = examples[0]
print("sai:", wrong_global_set(logs, k), "| đúng:", want)

sai: [1, 1, 0, 0, 0] | đúng: [0, 2, 0, 0, 0]


**Vì sao sai:** phút 2 của người 1 bị coi là "đã thấy" vì người 0 cũng hoạt động phút 2. Khử trùng phải theo **từng người** (hoặc theo cặp `(id, phút)`). Người dùng bị mất hết phút còn biến mất khỏi kết quả.

### Lỗi 4 — Trả về list dài bằng UAM lớn nhất thay vì `k`

In [22]:
def wrong_length(logs, k):
    d = defaultdict(set)
    for uid, t in logs:
        d[uid].add(t)
    uams = [len(ts) for ts in d.values()]
    ans = [0] * max(uams)                       # SAI: phải dài k
    for v in uams:
        ans[v - 1] += 1
    return ans

logs, k, want = examples[1]
print("sai:", wrong_length(logs, k), "| đúng:", want)

sai: [1, 1] | đúng: [1, 1, 0, 0]


**Vì sao sai:** đề yêu cầu mảng **kích thước `k`**, các ô cuối có thể bằng 0. **Sửa:** `ans = [0] * k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm số giá trị khác nhau theo từng nhóm"** ⇒ `defaultdict(set)` (hoặc `set` các cặp `(nhóm, giá trị)`).
- **"Đếm số nhóm có tần suất = j"** ⇒ "tần suất của tần suất": hai tầng đếm.
- Kết quả "1-indexed" ⇒ `ans[j − 1]`, độ dài theo đề (`k`), không theo dữ liệu.
- Kiểm nhanh: tổng các ô của `answer` = số người dùng khác nhau.

## 15. Biến thể của bài

1. **Trả về danh sách người theo UAM** thay vì số lượng (Bài tập 3).
2. **UAM trong một khoảng thời gian** `[L, R]` ⇒ lọc log trước; nhiều truy vấn ⇒ sắp xếp theo thời gian + cấu trúc đếm phân biệt (khó hơn nhiều).
3. **Chuỗi phút liên tiếp dài nhất** của mỗi người, rồi đếm theo độ dài chuỗi (Bài tập 5).
4. **Top-k người hoạt động nhiều nhất** ⇒ `heapq.nlargest` trên UAM.
5. Bài cùng mẫu "tần suất của tần suất": LeetCode 1207 *Unique Number of Occurrences*, 2287, 2404.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét lại | mỗi người quét hết log | `O(U·L)` | `O(L + k)` | chuẩn đối chiếu |
| 2. Sắp xếp | gom khúc, đếm phút khác | `O(L log L)` | `O(L + k)` | không dùng băm / cần thứ tự |
| 3. dict → set | nhóm + khử trùng | `O(L + k)` | `O(L + k)` | **nộp bài** |
| 4. set cặp + Counter | khử trùng cặp, đếm hai tầng | `O(L + k)` | `O(L + k)` | viết gọn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `logs = [[1,1],[1,1],[2,1],[2,2],[2,2]]`, `k = 3`?

**Bài 2 (tính tay):** Lập bảng `ID → tập phút` cho `logs = [[3,4],[3,5],[4,4],[3,4],[4,9],[4,10]]`, `k = 3`, rồi suy ra đáp án.

**Bài 3 (điền TODO):** Viết `users_by_uam(logs)` trả về `dict` `UAM → list ID đã sắp xếp tăng dần`.

**Bài 4 (tự viết lại):** Viết lại lời giải dùng **`groupby`** trên log đã sắp xếp (không tự quản lý `cur_user`).

**Bài 5 (mini challenge):** Với mỗi người, tính **độ dài chuỗi phút liên tiếp dài nhất** (ví dụ phút `{2,3,4,7}` cho 3). Trả về mảng dài `k` (giả sử `k ≥` độ dài lớn nhất), ô `j − 1` = số người có chuỗi dài nhất bằng `j`.

In [23]:
def users_by_uam(logs):
    # TODO
    pass

def uam_groupby(logs, k):
    # TODO
    pass

def longest_streak_hist(logs, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Người 1: `{1}` | Người 2: `{1, 2}` | `[1, 1, 0]` |
| 2 | Người 3: `{4, 5}` | Người 4: `{4, 9, 10}` | `[0, 1, 1]` |
| 3 | `defaultdict(set)` rồi `defaultdict(list)` | Nhóm theo `len(ts)` | `sorted(...)` mỗi danh sách |
| 4 | `groupby(sorted(logs), key=lambda p: p[0])` | Mỗi nhóm: `len({t for _, t in grp})` | `ans[uam − 1] += 1` |
| 5 | Sắp xếp tập phút của mỗi người | Đếm chuỗi: `t == prev + 1` thì tăng, ngược lại về 1 | Lấy max mỗi người, rồi đếm theo max |

## 19. Đáp án tham khảo

In [24]:
r1 = Solution().findingUsersActiveMinutes([[1, 1], [1, 1], [2, 1], [2, 2], [2, 2]], 3)
print("Bài 1:", r1); assert r1 == [1, 1, 0]
logs2 = [[3, 4], [3, 5], [4, 4], [3, 4], [4, 9], [4, 10]]
print("Bài 2:"); r2 = uam_dict_demo(logs2, 3); assert r2 == [0, 1, 1]

Bài 1: [1, 1, 0]
Bài 2:
log [3, 4]: phút của 3 = [4]
log [3, 5]: phút của 3 = [4, 5]
log [4, 4]: phút của 4 = [4]
log [3, 4]: phút của 3 = [4, 5]  (trùng, bỏ qua)
log [4, 9]: phút của 4 = [4, 9]
log [4, 10]: phút của 4 = [4, 9, 10]
người 3: UAM = 2 -> ans = [0, 1, 0]
người 4: UAM = 3 -> ans = [0, 1, 1]


In [25]:
def users_by_uam(logs):
    d = defaultdict(set)
    for u, t in logs:
        d[u].add(t)
    out = defaultdict(list)
    for u, ts in d.items():
        out[len(ts)].append(u)
    return {j: sorted(us) for j, us in sorted(out.items())}

def uam_groupby(logs, k):
    ans = [0] * k
    for _, grp in groupby(sorted(logs), key=lambda p: p[0]):
        ans[len({t for _, t in grp}) - 1] += 1
    return ans

rng = random.Random(3)
for _ in range(500):
    logs = [[rng.randint(0, 6), rng.randint(1, 6)] for _ in range(rng.randint(1, 25))]
    k = 6
    assert uam_groupby(logs, k) == uam_brute(logs, k)
    ub = users_by_uam(logs)
    assert [len(ub.get(j, [])) for j in range(1, k + 1)] == uam_brute(logs, k)
print("Bài 3, 4: OK", users_by_uam(examples[0][0]))

Bài 3, 4: OK {2: [0, 1]}


In [26]:
def longest_streak_hist(logs, k):
    d = defaultdict(set)
    for u, t in logs:
        d[u].add(t)
    ans = [0] * k
    for ts in d.values():
        best = cur = 0
        prev = None
        for t in sorted(ts):
            cur = cur + 1 if prev is not None and t == prev + 1 else 1
            best = max(best, cur)
            prev = t
        ans[best - 1] += 1
    return ans

def longest_streak_brute(logs, k):
    users = {u for u, _ in logs}
    ans = [0] * k
    for u in users:
        ts = {t for u2, t in logs if u2 == u}
        best = max(L for t in ts for L in range(1, len(ts) + 1) if all(t + d in ts for d in range(L)))
        ans[best - 1] += 1
    return ans

rng = random.Random(8)
for _ in range(500):
    logs = [[rng.randint(0, 4), rng.randint(1, 10)] for _ in range(rng.randint(1, 25))]
    assert longest_streak_hist(logs, 10) == longest_streak_brute(logs, 10)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", longest_streak_hist([[0, 2], [0, 3], [0, 4], [0, 7], [1, 5]], 4))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK [1, 0, 1, 0]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
minutes = defaultdict(set)
for uid, t in logs:
    minutes[uid].add(t)          # nhóm + khử trùng
ans = [0] * k
for ts in minutes.values():
    ans[len(ts) - 1] += 1        # 1-indexed -> 0-indexed
return ans
```

**Checklist:**
- [ ] Đếm **phút khác nhau** (dùng `set`), không đếm hành động?
- [ ] Khử trùng **theo từng người**, không dùng một tập phút chung?
- [ ] `ans[len(ts) - 1]` (trừ 1)?
- [ ] Độ dài kết quả đúng `k`?
- [ ] Tổng các ô = số người dùng khác nhau.